## SPARQL Execution
Use the existing grounding, generation, validation and bounded RDF execution pipeline.

In [ ]:
from pathlib import Path
from uuid import uuid4
import json
import os

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/hlo_recommender").is_dir())
os.chdir(ROOT)

from hlo_recommender.cli import parser, run
from hlo_recommender.io import read_jsonl, write_json, write_jsonl

SESSION_FILE = ROOT / "artifacts/notebooks/session.json"
SESSION = json.loads(SESSION_FILE.read_text()) if SESSION_FILE.exists() else {}


In [ ]:
from hlo_recommender.pipeline import Runtime

runtime = Runtime.from_file(ROOT / "configs/default.toml")
QUESTION = "Find resources for arterial blood gas interpretation."
CONTEXT = {"level": "Junior", "minutes": 15}
skills = runtime.grounder.skills(QUESTION)
print(runtime.grounder.schema_text(QUESTION))


### Generation
The template is the runnable baseline; select mT5 after completing neural fine-tuning.

In [ ]:
GENERATOR = "template"
CHECKPOINT = SESSION.get("mt5_checkpoint")
generator = runtime.generator(GENERATOR, CHECKPOINT)
retrieved = runtime.retrieve(QUESTION, CONTEXT, generator)
print(retrieved["query"])


### Validation
Inspect the project validator output and the resource identifiers returned by execution.

In [ ]:
{"status": retrieved["status"], "checks": retrieved.get("execution", {}).get("checks"), "resources": retrieved["resource_ids"]}
